In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4


In [ ]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token: ")

Paste your Kaggle API token: ··········


In [ ]:
!pip install -q --upgrade kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.8 MB/s eta 0:00:00


In [ ]:
!kaggle --version
!kaggle datasets files emmarex/plantdisease

Kaggle CLI 2.2.4
Next Page Token = CfDJ8JDwKpgsc5hJpcljKjpYpOFWjhatv7PwBlQwg7YeBn_gPaHc5lzFVxi1xiK4HED079IUyUnyGpVgyeDkyYjnEk9ccFAkY1RAxvHRKZRnHPn6GlVLBEQb_j85wusWnkSxyDsdEGJMu6rdVM0GT4yeqfJZbtbFv_tg1SLkob6f5cnJ2D4NMHdXFsoR4DLlcrm76puE5Y2c-f0O9yAfk7W7KRkU9zX7o-A3wtq-gpXAXOriCT2s3bEVwcI
name                                                                                                     size  creationDate                
------------------------------------------------------------------------------------------------------  -----  --------------------------  
PlantVillage/Pepper__bell___Bacterial_spot/0022d6b7-d47c-4ee2-ae9a-392a53f48647___JR_B.Spot 8964.JPG    18052  2019-10-10 14:49:43.555000  
PlantVillage/Pepper__bell___Bacterial_spot/006adb74-934f-448f-a14f-62181742127b___JR_B.Spot 3395.JPG    21211  2019-10-10 14:49:43.557000  
PlantVillage/Pepper__bell___Bacterial_spot/00f2e69a-1e56-412d-8a79-fdce794a17e4___JR_B.Spot 3132.JPG    18554  2019-10-10 14:49:43.523000  
PlantVillage/

In [ ]:
from pathlib import Path

data_dir = Path("/content/plant_data_full")
data_dir.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/plant_data_full

Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [00:28<00:00, 103MB/s]



In [ ]:
from pathlib import Path

data_dir = Path("/content/plant_data_full")

for file in data_dir.iterdir():
    size_mb = file.stat().st_size / (1024 ** 2) if file.is_file() else None

    if file.is_file():
        print(f"File: {file.name} | Size: {size_mb:.2f} MB")
    else:
        print(f"Folder: {file.name}")

File: new-plant-diseases-dataset.zip | Size: 2763.47 MB


In [ ]:
from pathlib import Path

data_dir = Path("/content/plant_data_full")

for folder_name in ["downloads", "plant_village"]:
  folder = data_dir / folder_name

print(f"\n📁 {folder_name}")

if folder.exists():
    for item in list(folder.iterdir())[:10]:
        print("Folder:" if item.is_dir() else "File:", item.name)
else:
    print("Folder not found")



📁 plant_village
Folder not found


In [ ]:
from pathlib import Path

data_dir = Path("/content/plant_data_full")

for path in data_dir.rglob("*"):
  if path.is_dir():
    print("📁", path)
  else:
    print("📄", path.name, f"({path.stat().st_size / (1024**2):.2f} MB)")


📄 new-plant-diseases-dataset.zip (2763.47 MB)


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/plant_data_full/new-plant-diseases-dataset.zip")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    files = zip_ref.namelist()

    print("Total files and folders:", len(files))
    print("\nFirst 30 items:")

    for item in files[:30]:
        print(item)

    image_files = [
        item for item in files
        if item.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print("\nTotal images inside ZIP:", len(image_files))

Total files and folders: 175767

First 30 items:
New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___Apple_scab/00075aa8-d81a-4184-8541-b692b78d398a___FREC_Scab 3335.JPG
New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___Apple_scab/01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Scab 3003.JPG
New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___Apple_scab/01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Scab 3003_270deg.JPG
New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___Apple_scab/01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Scab 3003_90deg.JPG
New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___Apple_scab/01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Scab 3003_new30degFlipLR.JPG
New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/Apple___Apple_scab/01f3deaa-6143-4b6c-9c22-620

In [ ]:
import zipfile
from pathlib import PurePosixPath
from collections import Counter
import pandas as pd

zip_path = "/content/plant_data_full/new-plant-diseases-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    image_paths = [
        name for name in z.namelist()
        if name.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

class_counts = Counter()
split_counts = Counter()

for path in image_paths:
    parts = PurePosixPath(path).parts

    # تحديد نوع الـ split
    if "train" in parts:
        split = "train"
    elif "valid" in parts:
        split = "valid"
    elif "test" in parts:
        split = "test"
    else:
        continue

    split_index = parts.index(split)

    # استخراج اسم الفئة
    if split_index + 1 < len(parts) - 1:
        class_name = parts[split_index + 1]
        class_counts[class_name] += 1
        split_counts[split] += 1

# الطباعة النهائية
print("Total images:", len(image_paths))
print("Number of classes:", len(class_counts))

print("\nImages per split:")
for split, count in split_counts.items():
    print(f"{split}: {count}")

# جدول منسق للفئات وعدد الصور
df = pd.DataFrame.from_dict(class_counts, orient="index", columns=["Image Count"])
df.index.name = "Class Name"
df = df.sort_index()

print("\nClass distribution table:")
print(df)


Total images: 175767
Number of classes: 39

Images per split:
train: 140590
valid: 35144
test: 33

Class distribution table:
                                                    Image Count
Class Name                                                     
Apple___Apple_scab                                         5040
Apple___Black_rot                                          4968
Apple___Cedar_apple_rust                                   4400
Apple___healthy                                            5020
Blueberry___healthy                                        4540
Cherry_(including_sour)___Powdery_mildew                   4208
Cherry_(including_sour)___healthy                          4564
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot         4104
Corn_(maize)___Common_rust_                                4768
Corn_(maize)___Northern_Leaf_Blight                        4770
Corn_(maize)___healthy                                     4648
Grape___Black_rot                          

In [ ]:
import zipfile
from pathlib import PurePosixPath
from collections import Counter

zip_path = "/content/plant_data_full/new-plant-diseases-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    image_paths = [
        p for p in z.namelist()
        if p.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

split_summary = Counter()
test_examples = []

for path in image_paths:
    parts = PurePosixPath(path).parts

    # تحديد نوع الـ split
    for split in ("train", "valid", "test"):
        if split in parts:
            split_index = parts.index(split)
            split_summary[split] += 1

            # نخزن أمثلة من test
            if split == "test" and len(test_examples) < 10:
                test_examples.append(path)

            break

print("Images per split:")
for split, count in split_summary.items():
    print(f"{split}: {count}")

print("\nExamples from test:")
for path in test_examples:
    print(path)


Images per split:
train: 140590
valid: 35144
test: 33

Examples from test:
test/test/AppleCedarRust1.JPG
test/test/AppleCedarRust2.JPG
test/test/AppleCedarRust3.JPG
test/test/AppleCedarRust4.JPG
test/test/AppleScab1.JPG
test/test/AppleScab2.JPG
test/test/AppleScab3.JPG
test/test/CornCommonRust1.JPG
test/test/CornCommonRust2.JPG
test/test/CornCommonRust3.JPG


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   46G   68G  40% /


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/plant_data_full/new-plant-diseases-dataset.zip")
extract_dir = Path("/content/plant_data_full/extracted")

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extraction completed!")
print("Dataset location:", extract_dir)

Extraction completed!
Dataset location: /content/plant_data_full/extracted


In [ ]:
from pathlib import Path

extract_dir = Path("/content/plant_data_full/extracted")

for split in ["train", "valid", "test"]:
    matches = [
        path for path in extract_dir.rglob(split)
        if path.is_dir()
    ]

    print(f"\n{split.upper()} folders:")
    for path in matches:
        print(path)



TRAIN folders:
/content/plant_data_full/extracted/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/train
/content/plant_data_full/extracted/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train

VALID folders:
/content/plant_data_full/extracted/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/valid
/content/plant_data_full/extracted/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid

TEST folders:
/content/plant_data_full/extracted/test
/content/plant_data_full/extracted/test/test


In [ ]:
from pathlib import Path
from torchvision.datasets import ImageFolder

train_dir = Path(
    "/content/plant_data_full/extracted/"
    "New Plant Diseases Dataset(Augmented)/"
    "New Plant Diseases Dataset(Augmented)/train"
)

valid_dir = Path(
    "/content/plant_data_full/extracted/"
    "New Plant Diseases Dataset(Augmented)/"
    "New Plant Diseases Dataset(Augmented)/valid"
)

print("Train exists:", train_dir.exists())
print("Valid exists:", valid_dir.exists())

if train_dir.exists() and valid_dir.exists():
    train_dataset = ImageFolder(train_dir)
    valid_dataset = ImageFolder(valid_dir)

    print("\nTraining images:", len(train_dataset))
    print("Validation images:", len(valid_dataset))

    print("\nTraining classes:", len(train_dataset.classes))
    print("Validation classes:", len(valid_dataset.classes))

    print("\nFirst 10 classes:")
    for class_name in train_dataset.classes[:10]:
        print(class_name)

    print(
        "\nSame class mapping:",
        train_dataset.class_to_idx == valid_dataset.class_to_idx
    )
else:
    print("Dataset path not found. Check the printed paths.")


Train exists: True
Valid exists: True

Training images: 70295
Validation images: 17572

Training classes: 38
Validation classes: 38

First 10 classes:
Apple___Apple_scab
Apple___Black_rot
Apple___Cedar_apple_rust
Apple___healthy
Blueberry___healthy
Cherry_(including_sour)___Powdery_mildew
Cherry_(including_sour)___healthy
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Corn_(maize)___Common_rust_
Corn_(maize)___Northern_Leaf_Blight

Same class mapping: True


In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU is not available. Enable T4 GPU from Colab Runtime settings."
    )


PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pathlib import Path

train_dir = Path(
    "/content/plant_data_full/extracted/"
    "New Plant Diseases Dataset(Augmented)/"
    "New Plant Diseases Dataset(Augmented)/train"
)

valid_dir = Path(
    "/content/plant_data_full/extracted/"
    "New Plant Diseases Dataset(Augmented)/"
    "New Plant Diseases Dataset(Augmented)/valid"
)

IMG_SIZE = 224
BATCH_SIZE = 32

weights_mean = [0.485, 0.456, 0.406]
weights_std = [0.229, 0.224, 0.225]

# Strong augmentation: forces the model to stop relying on the clean lab
# background / exact leaf position of the PlantVillage photos.
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(weights_mean, weights_std),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), value="random"),
])

valid_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(weights_mean, weights_std)
])

train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    root=valid_dir,
    transform=valid_transform
)

assert train_dataset.classes == valid_dataset.classes

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Training batches:", len(train_loader))
print("Validation batches:", len(valid_loader))
print("Batch size:", BATCH_SIZE)


Number of classes: 38
Training batches: 2197
Validation batches: 550
Batch size: 32


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# تحميل الموديل بأوزان مدربة مسبقًا
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# نجمد أول 14 بلوك فقط (الخصائص العامة: حواف، ألوان، أشكال)
# وبنفك آخر 5 بلوكات عشان الموديل يتعلم خصائص الأمراض نفسها
FROZEN_UNTIL = 14

for param in model.features.parameters():
    param.requires_grad = False

for param in model.features[FROZEN_UNTIL:].parameters():
    param.requires_grad = True

# استبدال طبقة التصنيف لتناسب فئات المشروع
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Model:", "MobileNetV2")
print("Classes:", num_classes)
print("Device:", device)
print(f"Trainable params: {trainable:,} / {total:,}")


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 145MB/s]


Model: MobileNetV2
Classes: 38
Device: cuda
Trainable params: 1,730,022 / 2,272,550


### Training (modified)
- Strong augmentation on train only
- Last 5 MobileNetV2 blocks unfrozen (lr 1e-4), classifier lr 1e-3
- Label smoothing, AdamW, cosine LR, mixed precision, best-checkpoint saving
- Valid accuracy here is optimistic (lab photos). Test on your own real photos below.

In [ ]:
import copy
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# label smoothing: يمنع الموديل من الثقة الزائدة (100%) في توقعاته
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# learning rate أقل للبلوكات اللي اتفكت، وأعلى للطبقة الجديدة
optimizer = AdamW([
    {"params": model.features[FROZEN_UNTIL:].parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3},
], weight_decay=1e-4)

EPOCHS = 5
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")

output_dir = Path("/content/green_lens_model")
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "green_lens_mobilenetv2.pth"

best_valid_loss = float("inf")
best_state = None

for epoch in range(EPOCHS):
    start = time.time()

    # مرحلة التدريب
    model.train()
    # البلوكات المجمدة تفضل eval عشان إحصائيات BatchNorm
    # ما تتعدلش على صور المعمل بس
    model.features[:FROZEN_UNTIL].eval()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    scheduler.step()

    train_loss /= train_total
    train_accuracy = 100 * train_correct / train_total

    # مرحلة التحقق
    model.eval()

    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            valid_loss += loss.item() * images.size(0)
            valid_correct += (outputs.argmax(dim=1) == labels).sum().item()
            valid_total += labels.size(0)

    valid_loss /= valid_total
    valid_accuracy = 100 * valid_correct / valid_total

    # نحفظ أفضل نسخة
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, model_path)
        saved = " (best, saved)"
    else:
        saved = ""

    print(f"\nEpoch {epoch + 1}/{EPOCHS}  [{time.time() - start:.0f}s]{saved}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2f}%")
    print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_accuracy:.2f}%")

# نرجع لأفضل أوزان قبل خطوة الحفظ التالية
model.load_state_dict(best_state)
print("\nLoaded best weights. Best valid loss:", round(best_valid_loss, 4))



Epoch 1/5  [633s] (best, saved)
Train Loss: 1.0571 | Train Acc: 90.71%
Valid Loss: 0.8205 | Valid Acc: 97.80%

Epoch 2/5  [604s] (best, saved)
Train Loss: 0.8557 | Train Acc: 96.68%
Valid Loss: 0.7726 | Valid Acc: 98.69%

Epoch 3/5  [601s] (best, saved)
Train Loss: 0.8146 | Train Acc: 97.72%
Valid Loss: 0.7492 | Valid Acc: 99.06%

Epoch 4/5  [605s] (best, saved)
Train Loss: 0.7930 | Train Acc: 98.25%
Valid Loss: 0.7419 | Valid Acc: 99.15%

Epoch 5/5  [605s] (best, saved)
Train Loss: 0.7836 | Train Acc: 98.39%
Valid Loss: 0.7324 | Valid Acc: 99.22%

Loaded best weights. Best valid loss: 0.7324


In [12]:
import json
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_dir = Path("/content/green_lens_model")

with open(model_dir / "class_names.json", "r", encoding="utf-8") as f:
    class_names = json.load(f)

model = models.mobilenet_v2(weights=None)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))

# تحميل أوزان الموديل
state_dict = torch.load(
    model_dir / "green_lens_mobilenetv2.pth",
    map_location=device
)

model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Model loaded successfully!")
print("Number of classes:", len(class_names))
print("Device:", device)


Model loaded successfully!
Number of classes: 38
Device: cuda


In [13]:
from pathlib import Path
import json
import torch

output_dir = Path("/content/green_lens_model")
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "green_lens_mobilenetv2.pth"
classes_path = output_dir / "class_names.json"

# حفظ أوزان الموديل
torch.save(model.state_dict(), model_path)

# حفظ أسماء الفئات بنفس ترتيب التدريب
with open(classes_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("Model saved:", model_path)
print("Classes saved:", classes_path)
print("Number of classes:", len(class_names))


Model saved: /content/green_lens_model/green_lens_mobilenetv2.pth
Classes saved: /content/green_lens_model/class_names.json
Number of classes: 38


In [14]:
print("Model device:", next(model.parameters()).device)
print("Number of classes:", len(class_names))
print("Model mode:", model.training)

print("\nTomato classes:")
for name in class_names:
    if "Tomato" in name:
        print(name)

Model device: cuda:0
Number of classes: 38
Model mode: False

Tomato classes:
Tomato___Bacterial_spot
Tomato___Early_blight
Tomato___Late_blight
Tomato___Leaf_Mold
Tomato___Septoria_leaf_spot
Tomato___Spider_mites Two-spotted_spider_mite
Tomato___Target_Spot
Tomato___Tomato_Yellow_Leaf_Curl_Virus
Tomato___Tomato_mosaic_virus
Tomato___healthy


In [ ]:
print("Model device:", next(model.parameters()).device)
print("Number of classes:", len(class_names))
print("Model mode:", model.training)

print("\nTomato classes:")
for name in class_names:
    if "Tomato" in name:
        print(name)

Model device: cuda:0
Number of classes: 38
Model mode: False

Tomato classes:
Tomato___Bacterial_spot
Tomato___Early_blight
Tomato___Late_blight
Tomato___Leaf_Mold
Tomato___Septoria_leaf_spot
Tomato___Spider_mites Two-spotted_spider_mite
Tomato___Target_Spot
Tomato___Tomato_Yellow_Leaf_Curl_Virus
Tomato___Tomato_mosaic_virus
Tomato___healthy


In [ ]:
import random
from pathlib import Path
from PIL import Image
import torch

class_name = "Tomato___Late_blight"
class_dir = valid_dir / class_name

image_files = list(class_dir.glob("*"))
image_path = random.choice(image_files)

image = Image.open(image_path).convert("RGB")

input_tensor = image_transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(input_tensor)
    probabilities = torch.softmax(outputs, dim=1)[0]

top_probs, top_indices = torch.topk(probabilities, 5)

print("Actual class:", class_name)
print("\nTop 5 predictions:")

for prob, idx in zip(top_probs, top_indices):
    print(f"{class_names[idx.item()]}: {prob.item() * 100:.2f}%")

Actual class: Tomato___Late_blight

Top 5 predictions:
Tomato___Late_blight: 88.45%
Potato___Late_blight: 3.73%
Tomato___Early_blight: 1.30%
Tomato___Septoria_leaf_spot: 0.66%
Cherry_(including_sour)___healthy: 0.44%


In [1]:
!pip install -q gradio

In [18]:
import os
import json
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import gradio as gr

# 1. إعداد الجهاز والمسارات
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dir = Path("/content/green_lens_model")
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "green_lens_mobilenetv2.pth"
classes_path = model_dir / "class_names.json"

# 2. تعريف قائمة الفئات الـ 38 الافتراضية
class_names = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]

# حفظ الملف احتياطياً على القرص
with open(classes_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

# 3. بناء هيكل الموديل وتحميل الأوزان المدربة
model = models.mobilenet_v2(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))

# التحقق من وجود ملف الأوزان أولاً
if not model_path.exists():
    raise FileNotFoundError(
        f"❌ لم يتم العثور على ملف أوزان النموذج في المسار: {model_path}\n"
        "الرجاء التأكد من إعادة تشغيل خلايا تدريب وحفظ الموديل أولاً، أو رفع ملف الأوزان (.pth) يدويًا إلى مجلد /content/green_lens_model/"
    )

state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

# 4. تحويلات الصورة للاختبار
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 5. دالة التوقع لـ Gradio (عرض أفضل 5 توقعات الآن)
def predict_disease(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")
    input_tensor = image_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    # قمنا بزيادة عدد الفئات المعروضة إلى 5، ويمكنك زيادتها أكثر إذا أردت
    top_probabilities, top_indices = torch.topk(probabilities, k=5)

    results = {}
    for prob, idx in zip(top_probabilities, top_indices):
        class_name = class_names[idx.item()]
        readable_name = class_name.replace("___", " - ").replace("_", " ")
        results[readable_name] = float(prob.item())

    return results

# 6. بناء واجهة Gradio وتشغيلها
interface = gr.Interface(
    fn=predict_disease,
    inputs=gr.Image(type="pil", label="Upload Plant Leaf Image"),
    outputs=gr.Label(num_top_classes=5, label="Predictions (Top 5 Classes)"),
    title="🍃 Green Lens Plant Disease Detector",
    description="Upload a photo of a plant leaf to identify potential diseases and confidence levels from the 38 trained classes.",
    theme="soft"
)

interface.launch(share=True, debug=True)

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://70368a9ed09aac320d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://70368a9ed09aac320d.gradio.live
